                # BootstrapFinetune (Apple Silicon / MPS)

                Bootstrap successful traces into training data, then update model weights rather than only prompt state.

                **Use it when:** You control a trainable model and want to distill accepted DSPy traces into a reusable local adapter.

                **What compilation changes:** A PEFT LoRA adapter for Qwen2.5-0.5B-Instruct; the prompt remains separately inspectable.

                Important configuration in this benchmark:

                - stock DSPy BootstrapFinetune with a Transformers/TRL provider boundary
- MPS selected on Apple Silicon; CPU remains an explicit fallback
- 18 full-profile training steps, batch size 1, LoRA rank 8, seed 42
- local self-teaching by default, so this row makes no OpenAI calls

                The 74-row dataset and pair-grouped train/validation/test membership are frozen.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import (
    artifact_paths,
    benchmark_snapshot,
    learned_program_preview,
    verify_prompt_artifact,
)

OPTIMIZER = 'bootstrap-finetune'
print(f"optimizer={OPTIMIZER!r}")
print("reading the checked-in chapter result; no API calls")

optimizer='bootstrap-finetune'
reading the checked-in chapter result; no API calls


                ## Compile shape

                This is the essential DSPy call used by the shared runner (setup variables omitted):

                ```python
                optimizer = dspy.BootstrapFinetune(
    metric=exact_match, train_kwargs=training_config,
    exclude_demos=True, num_threads=1,
)
optimized_detector = optimizer.compile(
    detector, trainset=trainset, teacher=local_teacher,
)
                ```

                `compile` returns a program. Calling that program on the untouched test examples is
                a separate phase; the notebook reports optimization cost/time separately from inference latency.

In [2]:
print(benchmark_snapshot(OPTIMIZER))
print()
print(artifact_paths(OPTIMIZER))

BootstrapFinetune — frozen full-profile rerun
status: completed
task model: Qwen/Qwen2.5-0.5B-Instruct
test accuracy: 50.0%
uplift: +0.0 points vs its Qwen run baseline
optimization: $0.0000 and 1.1min
inference latency: mean 1.16s; p95 1.88s
reload checks: prompt=True; model=True; predictions=3/3 labels
comparison boundary: same frozen split, separate Qwen/MPS experiment; not Luna-comparable

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/bootstrap-finetune.json
- canonical prompt: chapter06/results/final_prompts/bootstrap-finetune.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

Read score and adapter evidence together. This Qwen/MPS result uses the same frozen split but is not an absolute Luna-to-Luna comparison; follow the training path for device, loss, and adapter metadata.

The next cell shows a bounded readable preview. The complete, lossless prompt and
saved program snapshot remain at the paths printed above.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 0

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_state_equal': True, 'mismatches': []}


## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Evaluate the returned program on a test set that was not
used during compilation, and compare accuracy, compile cost, and inference latency
rather than treating a single score as the whole result.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.